
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 12</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Build a Medallion Architecture Pipeline</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Ingest raw data into Bronze, transform it into Silver, aggregate it into Gold, and explore how Databricks tracks the full pipeline.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment.

In [0]:
%run ./Includes/Classroom-Setup-1

**In this lesson:** You'll organize the process of getting data into tables into a production pattern.


<!-- LEARN: Medallion Architecture -->
<!-- Template: vertical-layered-stack (3 layers) -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">The Medallion Architecture</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">The Medallion Architecture organizes data into three layers. Each layer adds quality and structure, moving data from raw ingestion to business-ready analytics.</div>

<div style="display: flex; align-items: stretch; gap: 16px;">

<!-- Left arrow label -->
<div style="
    writing-mode: vertical-lr;
    transform: rotate(180deg);
    text-align: center;
    font-weight: 700;
    font-size: 14pt;
    color: #618794;
    padding: 0 6px;
    display: flex;
    justify-content: flex-end;
">
&larr; RAW TO REFINED
</div>

<!-- Stacked layers -->
<div style="flex: 1; display: flex; flex-direction: column; gap: 6px;">

<!-- Bronze -->
<div style="background: #CD7F32; color: white; border-radius: 8px 8px 4px 4px; padding: 22px 24px; text-align: center;">
  <div style="font-size: 18pt; font-weight: 700;">Bronze — Raw Ingestion</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">Data exactly as it arrived. No transformations, no filtering. The source of truth for what was received.</div>
</div>

<!-- Silver -->
<div style="background: #90A5B1; color: white; border-radius: 4px; padding: 18px 24px; text-align: center;">
  <div style="font-size: 16pt; font-weight: 700;">Silver — Cleaned &amp; Enriched</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">Data is cleaned, standardized, and enriched. Duplicates removed, types corrected, audit columns added. The foundation for analysis.</div>
</div>

<!-- Gold -->
<div style="background: #FFAB00; color: #0b2026; border-radius: 4px 4px 8px 8px; padding: 18px 24px; text-align: center;">
  <div style="font-size: 16pt; font-weight: 700;">Gold — Business-Ready</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.85;">Aggregated, filtered, or joined for specific business use cases. What dashboards and analysts consume.</div>
</div>

</div>

</div>

<!-- Key point callout -->
<div style="margin-top: 20px; padding: 16px 20px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Why three layers?</strong> Keeping raw data separate from cleaned data means you can always go back to the source. If a transformation has a bug, you fix the Silver logic and re-run it from Bronze. You never lose the original data.
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**The Medallion Architecture in practice**

- **Bronze** is your landing zone. Data arrives from files, APIs, streaming sources, or partner feeds and lands here with no changes. Even if the data has quality issues (missing values, wrong types, duplicates), it all goes into Bronze as-is.
- **Silver** is where you apply business rules. Standardize column names, cast data types, remove duplicates, add audit timestamps, filter out invalid records. This is the layer most data engineers spend their time building.
- **Gold** is purpose-built for consumers. A Gold table might aggregate daily revenue by region for a finance dashboard, or join customer and order data for a marketing report. You often have multiple Gold tables sourced from the same Silver tables.
- The architecture is not rigid. Some teams add a "Platinum" layer for ML features. Others skip Gold and let analysts query Silver directly. The core principle is the same: separate raw ingestion from transformation from consumption.

</details>

### Explore: Build the Bronze layer

First, let's confirm what files are available in the volume.

In [0]:
spark.sql(f"LIST '/Volumes/{my_catalog}/{my_schema}/myfiles/'").display()

Now create a Bronze table and load all CSV files into it using COPY INTO. Bronze is raw data with no transformations.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS current_employees_bronze (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
);

In [0]:
result = spark.sql(f"""
    COPY INTO current_employees_bronze
    FROM '/Volumes/{my_catalog}/{my_schema}/myfiles/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()

In [0]:
%sql
SELECT * 
FROM current_employees_bronze;

You should see **6 rows**: all employees from both CSV files, loaded as-is. This is your Bronze table; the raw, unmodified data.

### Explore: Build the Silver layer

Now transform the Bronze data into a cleaned Silver table. We'll standardize the `Role` column to uppercase and add audit timestamps so you know when each record was processed.

In [0]:
%sql
CREATE OR REPLACE TABLE current_employees_silver AS
SELECT
  ID,
  FirstName,
  Country,
  UPPER(Role) AS Role,
  current_timestamp() AS processed_timestamp,
  current_date() AS processed_date
FROM current_employees_bronze;

In [0]:
%sql
SELECT * 
FROM current_employees_silver;

Compare Silver to Bronze:
- **Role** is now uppercase (`Data Engineer` → `DATA ENGINEER`)
- Two new columns: **processed_timestamp** and **processed_date** - audit trail showing when the transformation ran
- Same 6 rows, but now cleaned and enriched

Silver is where most analysis starts. Bronze is your backup if you ever need to re-process from the original data.

---
### Explore: Build the Gold layer

Gold tables are what analysts and dashboards consume. They are aggregated, business-ready data built to answer a specific question.

Before writing the Gold table, let's define the aggregation logic in a temporary view. This counts employees by role.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_total_roles AS
SELECT
  Role,
  COUNT(*) AS TotalEmployees
FROM current_employees_silver
GROUP BY Role;

In [0]:
%sql
SELECT * 
FROM temp_total_roles;

The temp view shows the count of employees for each role. Now let's write this into a Gold table.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS total_roles_gold (
  Role STRING,
  TotalEmployees INT
);

In [0]:
%sql
INSERT OVERWRITE total_roles_gold
SELECT * FROM temp_total_roles;

In [0]:
%sql
SELECT * 
FROM total_roles_gold;

You now have a Gold table that answers a specific business question: "How many employees do we have in each role?" This is what a dashboard or report would query.

The `INSERT OVERWRITE` pattern means you can refresh this table by re-running the same command. It replaces the data completely with the latest aggregation from Silver.

### Explore: Check the version history

In [0]:
%sql
DESCRIBE HISTORY total_roles_gold;

---
### Explore: View lineage and governance in Catalog Explorer

Now that you've built the full pipeline, let's see how Databricks tracks it automatically.

**Follow these steps:**

1. In the left sidebar, click **Catalog**
2. Navigate to `labuser` → `get_started_de` → **Tables** → `total_roles_gold`
3. Click the **Lineage** tab → click **See lineage graph**
   - You should see the chain: CSV files → Bronze → Silver → Gold
4. Click the **Permissions** tab
   - Click **Grant** to see the available permission options, then click **Cancel**
   - In production, this is how you'd control who can query each table
5. Click the **Insights** tab
   - This shows recent query activity on the table
   - In a production environment, this helps you understand which tables are actively being used

*Note: Lineage data may take a few minutes to appear. If the lineage graph isn't visible yet, check back shortly.*


##### EXPAND FOR ADDITIONAL NOTES

<details>

**Gold tables in practice**

- Gold tables are purpose-built. While Bronze and Silver are shared infrastructure, Gold tables are often built for a specific team, dashboard, or report.
- A common pattern is using `INSERT OVERWRITE` instead of `CREATE OR REPLACE`. This lets you define the Gold table once (with the right schema and permissions) and then refresh its contents on a schedule without recreating the table.
- You might have multiple Gold tables sourced from the same Silver table: one for finance (revenue aggregations), one for HR (headcount by department), one for ops (SLA metrics).

**Data governance features**

- **Lineage** shows the full chain: which notebooks, tables, and files feed into a given table. If the Gold table looks wrong, lineage helps you trace back to the source.
- **Permissions** control who can SELECT, MODIFY, or manage each table. In production, you'd grant analysts SELECT on Gold tables but restrict access to Bronze.
- **Insights** show query activity: who's querying the table, how often, and what kinds of queries they run. Useful for understanding which tables are actually being used.

</details>


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Created a <strong>Bronze</strong> table with raw data using <code>COPY INTO</code></li>
      <li>Created a <strong>Silver</strong> table with transformations (<code>UPPER</code>, audit timestamps)</li>
      <li>Created a <strong>Gold</strong> table with aggregated, business-ready data</li>
      <li>Used the <code>INSERT OVERWRITE</code> pattern for refreshable Gold tables</li>
      <li>Explored <strong>lineage</strong>, <strong>permissions</strong>, and <strong>insights</strong> in Catalog Explorer</li>
    </ul>
    <div style="margin-top: 12px;">You've built a complete Medallion Architecture pipeline by hand. </div>
  </div>
</div>
</div>


<!-- CHECKPOINT: Lesson 12 -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 24px 28px; text-align: center;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Checkpoint</div>
  <div style="font-size: 20pt; font-weight: 700;">What You've Done So Far</div>
</div>

<div style="margin-top: 16px; padding: 20px 24px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.7;">
    <p>In this lesson, you built a complete Medallion Architecture pipeline:</p>
    <ul style="padding-left: 20px; margin: 8px 0;">
      <li><strong>Bronze</strong> — raw data ingested from CSV files with <code>COPY INTO</code></li>
      <li><strong>Silver</strong> — cleaned and enriched with transformations and audit timestamps</li>
      <li><strong>Gold</strong> — aggregated for business consumption with <code>INSERT OVERWRITE</code></li>
    </ul>
    <p>You also explored how Unity Catalog automatically tracks lineage, permissions, and usage across all three layers.</p>
  </div>
</div>

<div style="margin-top: 16px; padding: 16px 20px; background: #F8F9FC; border-left: 4px solid #1B5162; border-radius: 6px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Quick self-check:</strong> Could you explain to a teammate why data goes through three layers instead of loading it straight into a final table?
  </div>
</div>



&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>